## Air Flight Booking Agent with OpenAI

This notebook uses air flight booking examples to demonstrate a simple tool-using agent.

In [1]:
!uv add openai

Resolved 113 packages in 2.32s                                       
Prepared 1 package in 1.94s                                              
Installed 6 packages in 23ms                                
 + httpcore2==2.12.0
 + httpx2==2.12.0
 + jiter==0.16.0
 + openai==3.13.0
 + sniffio==1.3.1
 + truststore==0.10.4


In [2]:
!uv add python-dotenv

Resolved 113 packages in 5ms
Audited 109 packages in 4ms


In [4]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# https://platform.openai.com/api-keys
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Set OPENAI_API_KEY in your .env file before running this notebook.")

client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-5-nano"

In [5]:
output = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Find me a flight from Entebbe to Nairobi on 2026-10-15."},
    ],
    stream=False,
    reasoning_effort="minimal",
    max_completion_tokens=1024,
)
print(output.choices[0].message.content)

I can help plan this, but I don’t have live flight data or booking access. Here are a few practical options to find and book a flight from Entebbe (EBB) to Nairobi (NBO) on 2026-10-15:

Option A: Quick search steps (self-serve)
- Use flight search engines: Google Flights, Skyscanner, Kayak, Momondo, or Expedia.
- Enter: From: Entebbe International Airport (EBB), To: Nairobi (NBO). Date: 2026-10-15.
- Filter by: Nonstop flights, preferred times, airlines, and price range.
- Popular nonstop options typically include Kenya Airways (KQ) and Uganda Airlines (if operating on that date), and various Ethiopian/Egyptian carriers sometimes route via add-ons but direct is common.
- Check baggage rules, takeoff/landing times, and total travel time.

Option B: Airlines and carriers to check
- Kenya Airways (KQ) – often nonstop EBB to NBO
- Uganda Airlines – may offer EBB to NBO routes if operating
- Other carriers with 1-stop via Addis Ababa (Lagos? Not typical), or via Dubai/Doha depending on seas

In [6]:
SYSTEM_PROMPT = """Answer air flight booking questions as best you can. You have access to the following tool:

search_flights: Find available flights for a route and departure date.

Use the tool by specifying a JSON object with an `action` key for the tool name and an `action_input` key for its inputs.

The only value allowed in the `action` field is:
search_flights: Find available flights, args: {"origin": {"type": "string"}, "destination": {"type": "string"}, "departure_date": {"type": "string"}}

Example use:

{
  "action": "search_flights",
  "action_input": {"origin": "Entebbe", "destination": "Nairobi", "departure_date": "2026-10-15"}
}

ALWAYS use the following format:

Question: the air flight booking question you must answer
Thought: think about one action to take. Only one action at a time in this format:
Action:

$JSON_BLOB (inside a markdown code block)

Observation: the result of the action. This Observation is unique, complete, and the source of truth.
After outputting one Action, stop and wait for the application to provide the Observation.
Never invent an Observation or a Final Answer before the application provides the Observation.
(This Thought/Action/Observation cycle can repeat as needed. The $JSON_BLOB must be formatted as markdown and use only a SINGLE action at a time.)

After the application provides an Observation, end your output with the following format:

Thought: I now know the final answer
Final Answer: the final answer to the original air flight booking question

Now begin! Reminder: ALWAYS use the exact characters `Final Answer:` when you provide a definitive answer. """

In [7]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Find flights from Entebbe to Nairobi on 2026-10-15."},
]

print(messages)

[{'role': 'system', 'content': 'Answer air flight booking questions as best you can. You have access to the following tool:\n\nsearch_flights: Find available flights for a route and departure date.\n\nUse the tool by specifying a JSON object with an `action` key for the tool name and an `action_input` key for its inputs.\n\nThe only value allowed in the `action` field is:\nsearch_flights: Find available flights, args: {"origin": {"type": "string"}, "destination": {"type": "string"}, "departure_date": {"type": "string"}}\n\nExample use:\n\n{\n  "action": "search_flights",\n  "action_input": {"origin": "Entebbe", "destination": "Nairobi", "departure_date": "2026-10-15"}\n}\n\nALWAYS use the following format:\n\nQuestion: the air flight booking question you must answer\nThought: think about one action to take. Only one action at a time in this format:\nAction:\n\n$JSON_BLOB (inside a markdown code block)\n\nObservation: the result of the action. This Observation is unique, complete, and t

In [8]:
output = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    stream=False,
    reasoning_effort="minimal",
    max_completion_tokens=2000,
)
print(output.choices[0].message.content)

Question: Find flights from Entebbe to Nairobi on 2026-10-15.

Thought: I will search for available flights for the specified route and date.

Action:
{
  "action": "search_flights",
  "action_input": {
    "origin": "Entebbe",
    "destination": "Nairobi",
    "departure_date": "2026-10-15"
  }
}


In [9]:
# Dummy air flight search function
def search_flights(origin, destination, departure_date):
    return (
        f"Available flights from {origin} to {destination} on {departure_date}:\n"
        "1. Uganda Airlines UR 204, 09:00-10:15, USD 245\n"
        "2. Kenya Airways KQ 419, 16:20-17:35, USD 278\n"
    )

search_flights("Entebbe", "Nairobi", "2026-10-15")

'Available flights from Entebbe to Nairobi on 2026-10-15:\n1. Uganda Airlines UR 204, 09:00-10:15, USD 245\n2. Kenya Airways KQ 419, 16:20-17:35, USD 278\n'

In [13]:
action_response = output.choices[0].message.content
flight_results = search_flights(
    "Entebbe",
    "Nairobi",
    "2026-10-15",
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Find flights from Entebbe to Nairobi on 2026-10-15."},
    {"role": "assistant", "content": action_response},
    {
        "role": "user",
        "content": (
            f"Observation:\n{flight_results}\n"
            "Use this observation to provide the Final Answer."
        ),
    },
]

final_output = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    stream=False,
    reasoning_effort="minimal",
    max_completion_tokens=1000,
)

print(final_output.choices[0].message.content)

Question: Find flights from Entebbe to Nairobi on 2026-10-15.

Thought: I now have the flight options from the Observation. I will present them clearly and offer next steps for booking.

Observation: Available flights from Entebbe to Nairobi on 2026-10-15:
1. Uganda Airlines UR 204, 09:00-10:15, USD 245
2. Kenya Airways KQ 419, 16:20-17:35, USD 278

Thought: I now know the final answer
Final Answer: Here are the available flights from Entebbe to Nairobi on 2026-10-15:
- Uganda Airlines UR 204, depart 09:00, arrive 10:15, price USD 245
- Kenya Airways KQ 419, depart 16:20, arrive 17:35, price USD 278

Would you like me to book one of these or provide more details (baggage, seat maps, cancellations)?
